In [1]:
from pyspark.sql import SparkSession
# .config("spark.executor.memory", "4g")
spark = SparkSession.builder.config("spark.driver.memory", "100g").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/28 18:12:59 WARN Utils: Your hostname, mark-desktop, resolves to a loopback address: 127.0.1.1; using 192.168.4.20 instead (on interface eno1)
26/02/28 18:12:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 18:13:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import os

os.getcwd()

'/home/mark/Documents/projects/arxiv_analysis/exploration'

In [7]:
from pathlib import Path
from pyspark.sql.functions import *
from pyspark.sql.types import *

open_alex_data_dir = Path(os.getcwd()).parent / 'data' / 'open_alex' / 'openalex-snapshot' / 'data'

authors_dir = open_alex_data_dir / 'merged_ids' / 'authors'
institutions_dir = open_alex_data_dir / 'merged_ids' / 'institutions'
works_dir = open_alex_data_dir / 'works'

In [8]:
http_prefix = 'https://openalex.org/'

def get_code(col_name: str):
    return col(col_name).substr(lit(len(http_prefix) + 1), char_length(col(col_name)))


In [9]:
open_alex_data_dir

PosixPath('/home/mark/Documents/projects/arxiv_analysis/data/open_alex/openalex-snapshot/data')

In [10]:
authors_df = spark.read.json(str(authors_dir))
# authors_df.show()

In [11]:
authors_df.schema

StructType([StructField('_corrupt_record', StringType(), True)])

In [12]:
authors_df.printSchema()

root
 |-- _corrupt_record: string (nullable = true)



In [19]:
http_prefix = 'https://openalex.org/'

authors_df.select(col('id'), col('id').substr(lit(len(http_prefix) + 1), char_length(col('id')))).head()

Row(id='https://openalex.org/A5056093285', substring(id, 22, char_length(id))='A5056093285')

In [9]:
http_prefix = 'https://openalex.org/'

def get_code(col_name: str):
    return col(col_name).substr(lit(len(http_prefix) + 1), char_length(col(col_name)))

authors_table_df = (
    authors_df
    .withColumn('id_code', get_code('id'))
    .withColumn('last_known_institution_id', get_code('last_known_institution.id'))
    .select(
        'id_code',
        'display_name',
        'works_count',
        'cited_by_count',
        'most_cited_work',
        'summary_stats',
        'last_known_institution_id',
        'last_known_institution',
        'updated_date',
        'created_date',
        'updated',
    )
)
# authors_table_df = authors_table_df.repartition(48)
authors_table_df.repartition(24).write.option('compression', 'zstd').parquet(str(open_alex_data_dir / 'authors_parquet'))

In [30]:
57_382 / 290

197.86896551724138

In [31]:
29 % 24

5

In [33]:
(128 - 20) / 24

4.5

In [7]:
2339/24

48.729166666666664

In [14]:
single_work_dir = works_dir / 'updated_date=2024-01-22'
single_work_dir.exists()

True

# Works

In [15]:
works_df = spark.read.json(str(single_work_dir))

In [16]:
all_works_df = spark.read.json(str(works_dir), schema=works_df.schema)

In [17]:
works_df.printSchema()

root
 |-- abstract_inverted_index: string (nullable = true)
 |-- apc_list: string (nullable = true)
 |-- apc_paid: string (nullable = true)
 |-- authors_count: long (nullable = true)
 |-- authorships: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- author: struct (nullable = true)
 |    |    |    |-- display_name: string (nullable = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- orcid: string (nullable = true)
 |    |    |-- author_position: string (nullable = true)
 |    |    |-- countries: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- institutions: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- country_code: string (nullable = true)
 |    |    |    |    |-- display_name: string (nullable = true)
 |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |-- lineage: array (nullable = true)
 | 

In [18]:
import json
with open(open_alex_data_dir / 'works_spark_schema', 'w') as f:
    json.dump(works_df.schema.json(), f)

In [19]:
workers_table_df = (
    all_works_df
    .withColumn('id_code', get_code('id'))
    .withColumn('source_id_code', get_code('primary_location.source.id'))
    .select(
        'id_code',
        'doi',
        'doi_registration_agency',
        'display_name',
        'title',
        'publication_date',
        col('primary_location.is_accepted').alias('is_accepted'),
        col('primary_location.is_published').alias('is_published'),
        col('source_id_code'),
        'language',
        'concepts_count',
        'authors_count',
        'authorships',
        'referenced_works_count',
        'referenced_works',
        'related_works',
        'cited_by_count',
        'cited_by_percentile_year',
        'counts_by_year',
        'updated_date',
        'created_date',
        'updated',
    )
)
workers_table_df.coalesce(72).write.option('compression', 'zstd').parquet(str(open_alex_data_dir / 'works_parquet'))

In [21]:
363653 / 5_000

72.7306

In [22]:
24*3

72

In [25]:
all_works_df = spark.read.parquet(str(open_alex_data_dir / 'works_parquet')).repartition(48)
all_works_df.write.option('compression', 'zstd').parquet(str(open_alex_data_dir / 'works_parquet_partitioned'))

In [20]:
merged_ids_dir = open_alex_data_dir / 'merged_ids'
data_types = [
    'authors',
    'sources',
    'works',
    'institutions',
    'publishers',
]
merged_dir_tuples = [(merged_ids_dir / data_type, data_type) for data_type in data_types]

In [21]:
merged_df_tuples = []
for dir, data_type in merged_dir_tuples:
    d_type_df = spark.read.csv(str(dir), header=True, inferSchema=True)
    print('data_type', data_type)
    d_type_df.printSchema()
    merged_df_tuples.append((d_type_df, data_type))

data_type authors
root
 |-- merge_date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- merge_into_id: string (nullable = true)

data_type sources
root
 |-- merge_date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- merge_into_id: string (nullable = true)



data_type works
root
 |-- merge_date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- merge_into_id: string (nullable = true)

data_type institutions
root
 |-- merge_date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- merge_into_id: string (nullable = true)

data_type publishers
root
 |-- merge_date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- merge_into_id: string (nullable = true)



In [22]:
merged_df = None
for i, tuple in enumerate(merged_df_tuples):
    df, data_type = tuple
    df_with_data_type = df.withColumn("data_type", lit(data_type))
    if i == 0:
        merged_df = df_with_data_type
    else:
        merged_df = merged_df.union(df_with_data_type)

In [23]:
merged_df.write.parquet(str(open_alex_data_dir / 'merged_ids_parquet'))